# DualLens Analytics

**Ranking companies for investment by fusing two lenses — quantitative financial performance and qualitative AI-initiative strategy — through Retrieval-Augmented Generation.**

Traditional analysis looks at financial metrics alone and misses how *future-ready* a company is. Strategy documents hold that qualitative signal but are hard to query and fuse with numbers. DualLens closes that gap: stock trends provide evidence of momentum, AI-initiative reports reveal innovation capacity, and an LLM ranks companies on both.

| Lens | Source | Tools |
|------|--------|-------|
| **Quantitative** | Yahoo Finance (3-yr prices + metrics) | `yfinance`, `pandas`, `matplotlib` |
| **Qualitative** | 5 company AI-initiative PDFs | RAG: OpenAI embeddings + ChromaDB |
| **Reasoning** | RAG answers, LLM-as-Judge, ranking | **Claude Opus 4.8** (`claude-opus-4-8`) |

> **Note on the stack:** Claude has no embeddings API, so OpenAI (`text-embedding-3-small`) handles embeddings while Claude Opus 4.8 does all reasoning. Companies analysed: Amazon, Alphabet, IBM, Microsoft, NVIDIA.

## 1. Setup & Imports

Keys load from a local `.env` (gitignored). Copy `.env.example` to `.env` and fill in `ANTHROPIC_API_KEY` and `OPENAI_API_KEY` first.

In [ ]:
import os, json, re
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Load keys from .env at the repo root (one level up from duallens/)
load_dotenv(dotenv_path=Path('..') / '.env')
load_dotenv()  # also try current dir

assert os.getenv('ANTHROPIC_API_KEY'), 'Set ANTHROPIC_API_KEY in .env'
assert os.getenv('OPENAI_API_KEY'),   'Set OPENAI_API_KEY in .env'

LLM_MODEL   = 'claude-opus-4-8'          # all reasoning
EMBED_MODEL = 'text-embedding-3-small'   # embeddings only (Claude has none)

COMPANIES = {
    'AMZN': 'Amazon',
    'GOOGL': 'Alphabet (Google)',
    'IBM': 'IBM',
    'MSFT': 'Microsoft',
    'NVDA': 'NVIDIA',
}
print('Setup OK —', len(COMPANIES), 'companies')

## 2. Quantitative Lens — Stock Performance & Financials

### 2a. Three-Year Closing Prices
Pull the last 3 years of daily closing prices for each company and plot them together.

In [ ]:
closes = {}
for ticker, name in COMPANIES.items():
    hist = yf.Ticker(ticker).history(period='3y')
    closes[name] = hist['Close']

prices = pd.DataFrame(closes)

fig, ax = plt.subplots(figsize=(11, 5))
for name in prices.columns:
    ax.plot(prices.index, prices[name], label=name, linewidth=1.5)
ax.set_title('3-Year Closing Price', fontsize=13)
ax.set_ylabel('Price (USD)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Normalised view (growth from a common base of 100) — fairer momentum comparison
normed = prices / prices.iloc[0] * 100
fig, ax = plt.subplots(figsize=(11, 5))
for name in normed.columns:
    ax.plot(normed.index, normed[name], label=name, linewidth=1.5)
ax.set_title('3-Year Growth (indexed to 100)', fontsize=13)
ax.set_ylabel('Indexed price')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 2b. Financial Metrics
Market Cap, P/E ratio, Dividend Yield, Beta, and Total Revenue for each company.

In [ ]:
METRICS = {
    'Market Cap': 'marketCap',
    'P/E Ratio': 'trailingPE',
    'Dividend Yield': 'dividendYield',
    'Beta': 'beta',
    'Total Revenue': 'totalRevenue',
}

rows = {}
for ticker, name in COMPANIES.items():
    info = yf.Ticker(ticker).info
    rows[name] = {label: info.get(key) for label, key in METRICS.items()}

metrics_df = pd.DataFrame(rows).T
metrics_df

In [ ]:
fig, axes = plt.subplots(1, len(METRICS), figsize=(18, 3.5))
for ax, col in zip(axes, metrics_df.columns):
    metrics_df[col].plot(kind='bar', ax=ax, color='#0f766e')
    ax.set_title(col, fontsize=11)
    ax.tick_params(axis='x', rotation=45, labelsize=8)
plt.tight_layout()
plt.show()

## 3. Qualitative Lens — RAG over AI-Initiative PDFs

### 3a. Load & Chunk the PDFs
Load the 5 company AI-initiative PDFs and split them into overlapping chunks.

In [ ]:
PDF_DIR = next((p for p in [Path('Companies-AI-Initiatives'), Path('..') / 'Companies-AI-Initiatives'] if p.exists()), None)
assert PDF_DIR, 'Companies-AI-Initiatives/ not found'

docs = PyPDFDirectoryLoader(str(PDF_DIR)).load()

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
chunks = splitter.split_documents(docs)
print(f'Loaded {len(docs)} pages -> {len(chunks)} chunks from {PDF_DIR}')

### 3b. Embed & Store in ChromaDB
Embed each chunk with OpenAI `text-embedding-3-small` and persist them in a local Chroma store.

In [ ]:
embeddings = OpenAIEmbeddings(model=EMBED_MODEL)

vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name='duallens_ai_initiatives',
    persist_directory='chroma_duallens',
)
retriever = vectordb.as_retriever(search_kwargs={'k': 4})
print('Vector store ready:', vectordb._collection.count(), 'chunks indexed')

### 3c. Retrieval-Augmented Answering (Claude Opus 4.8)
Retrieve the most relevant chunks and let Claude answer, grounded strictly in the retrieved context.

In [ ]:
llm = ChatAnthropic(model=LLM_MODEL, max_tokens=1500)  # no temperature — Opus 4.8 rejects it

RAG_PROMPT = ChatPromptTemplate.from_template(
    'You are a financial research assistant. Answer the question using ONLY the context below.\n'
    "If the context does not contain the answer, say so. Cite the company name(s) you draw from.\n\n"
    'Context:\n{context}\n\nQuestion: {question}\n\nAnswer:'
)

def format_docs(retrieved):
    return '\n\n'.join(
        f"[{Path(d.metadata.get('source', '?')).stem}] {d.page_content}" for d in retrieved
    )

rag_chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

sample_q = 'Which companies are investing most heavily in generative AI, and how?'
answer = rag_chain.invoke(sample_q)
print(answer)

### 3d. Evaluation — LLM-as-Judge (Groundedness + Relevance)
Score the RAG answer on **groundedness** (is every claim supported by the retrieved context?) and **relevance** (does it address the question?), 1–5 each.

In [ ]:
judge_llm = ChatAnthropic(model=LLM_MODEL, max_tokens=500)

JUDGE_PROMPT = ChatPromptTemplate.from_template(
    'You are an impartial evaluator. Given a QUESTION, the retrieved CONTEXT, and an ANSWER, '
    'score the answer from 1 (poor) to 5 (excellent) on:\n'
    '- groundedness: is every claim supported by the context?\n'
    '- relevance: does it address the question?\n'
    'Return ONLY JSON: {{"groundedness": int, "relevance": int, "rationale": "one sentence"}}\n\n'
    'QUESTION: {question}\nCONTEXT: {context}\nANSWER: {answer}'
)

def extract_json(text):
    match = re.search(r'\{.*\}', text, re.DOTALL)
    return json.loads(match.group(0)) if match else {'error': text}

def evaluate(question):
    context = format_docs(retriever.invoke(question))
    answer = rag_chain.invoke(question)
    verdict = judge_llm.invoke(JUDGE_PROMPT.format_messages(question=question, context=context, answer=answer))
    return answer, extract_json(verdict.content)

ans, scores = evaluate(sample_q)
print('Scores:', scores)

## 4. Fusion — Scoring & Ranking (Quant + Qual)
Give Claude both lenses — the quantitative metrics table and a per-company qualitative summary from RAG — and have it rank the companies for investment with a Buy / Hold / Sell call and reasoning.

In [ ]:
# One qualitative summary per company, grounded in its PDF
qual_insights = {}
for name in COMPANIES.values():
    qual_insights[name] = rag_chain.invoke(
        f"In 3 concise bullets, summarise {name}'s AI initiatives and future readiness."
    )

qual_block = '\n\n'.join(f'### {name}\n{text}' for name, text in qual_insights.items())
print('Collected qualitative summaries for', len(qual_insights), 'companies')

In [ ]:
RANK_PROMPT = ChatPromptTemplate.from_template(
    'You are an investment analyst. Rank these companies for a long-term investor by fusing '
    'QUANTITATIVE financial data with QUALITATIVE AI-initiative insight. For each company give a '
    'Buy / Hold / Sell call and one-line reasoning that references BOTH a number and an AI initiative. '
    'End with the single best pick and why.\n\n'
    'QUANTITATIVE METRICS:\n{metrics}\n\n'
    'QUALITATIVE AI INITIATIVES:\n{qual}\n'
)

ranking = (RANK_PROMPT | llm | StrOutputParser()).invoke(
    {'metrics': metrics_df.to_string(), 'qual': qual_block}
)
print(ranking)

## 5. Summary & Future Scope

**What this builds:** a dual-lens investment screener that fuses quantitative momentum (3-yr prices, Market Cap, P/E, Beta, Revenue) with qualitative AI-strategy signal retrieved from company reports, and produces a grounded, cited ranking — with the RAG step independently graded by an LLM-as-Judge for groundedness and relevance.

**Future scope:**
1. Expand the evaluation set with a labelled gold-standard Q&A set and report average groundedness/relevance across many questions.
2. Add hybrid retrieval (dense + keyword) and a re-ranker to raise retrieval precision.
3. Broaden the universe beyond 5 companies and refresh financials on a schedule.